# MobileNetV3-Small × CRNN 85 epoch — Thực nghiệm Mục 4.5

**Mục tiêu:** Đánh giá tác động của **nguồn dự đoán bố cục** đối với OCR bằng **cùng CRNN–BiLSTM–CTC epoch 85**, cùng **5.649 ảnh test** và cùng nhãn. Ba nhánh:

1. **Reference layout → CRNN**: nhãn bố cục đã hiệu chỉnh của VNLP.
2. **Geometry (ngưỡng W/H) → CRNN**: ngưỡng lựa chọn **chỉ bằng validation**.
3. **MobileNetV3-Small → CRNN**: MobileNet được **huấn luyện lại trên train của OCR 70/15/15**, không dùng model MobileNet 80/10/10 để dự đoán test 70/15/15.

**Không sửa notebook CRNN hoặc model MobileNet cũ.** Checkpoint MobileNet và bảng kết quả mới nằm ở thư mục độc lập. **Không huấn luyện lại CRNN.**

> Bảo đảm tính khoa học: notebook chỉ tạo Bảng 4.7 khi tất cả 5.649 `image_id` đúng nhãn, CRNN epoch 85 và **toàn bộ dự đoán CRNN của nhánh reference khớp với `crnn_test_predictions.csv` đã lưu**. Không tự điền số liệu khi chưa chạy Colab.

## 0. Thư viện và kết nối Google Drive

Chọn **Runtime → Change runtime type → T4 GPU**, chạy các cell theo thứ tự. Đổi đường dẫn ở cell kế tiếp nếu Drive của bạn khác cấu trúc từ notebook CRNN.

In [ ]:
%pip -q install pandas numpy opencv-python-headless pillow matplotlib tqdm scikit-learn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Cấu hình — đường dẫn và quy tắc đánh giá

In [ ]:
from pathlib import Path
PROJECT = Path('/content/drive/MyDrive/Computer Vision/DoAnCuoiKy')
CRNN_DIR = PROJECT / 'CRNN_Retrain_LPRNet_v5_SameSplit'
ZIP_PATH = PROJECT / 'Dataset/VNLP_detection.zip'
MANIFEST_PATH = CRNN_DIR / 'shared_manifest_verified.csv'
CRNN_CHECKPOINT = CRNN_DIR / 'crnn_best.pt'
CRNN_TEST_CSV = CRNN_DIR / 'crnn_test_predictions.csv'

# Chỉ ghi vào output mới. Model MobileNet 80/10/10 và checkpoint CRNN không bị đụng tới.
OUTPUT_DIR = PROJECT / 'MobileNetV3_Layout_for_CRNN85_70_15_15_Chuong4_5'
LOCAL_DATA = Path('/content/vnlp_layout_70_15_15')

SEED=42
BATCH_SIZE=64
NUM_WORKERS=2
MAX_EPOCHS=25
WARMUP_EPOCHS=2
EARLY_STOP_PATIENCE=6
EVAL_BATCH=32
TRAIN_IF_NEEDED=True     # False = chỉ đánh giá nếu đã có layout_best.pt
COPY_ZIP_TO_LOCAL=True   # tránh 37k lần đọc file ảnh nhỏ trực tiếp trên Google Drive
REQUIRE_GPU=True

# Giao thức chính xác của CRNN epoch 85
CRNN_W,CRNN_H=256,32
IMG_W,IMG_H=94,24
PAD_X,PAD_Y=.03,.06
SPLIT_METHOD='sloped'
SPLIT_OVERLAP=.03
LABELS=['one_row','two_rows']
REFERENCE_EXPECTED={'total':37297,'train':26079,'val':5510,'test':5649,'epoch':85}
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
print('CRNN:', CRNN_DIR)
print('Output mới:', OUTPUT_DIR)

## 2. Nạp và đối chiếu manifest với CRNN epoch 85

Bản kê gốc có **37.297 ảnh**; tập **train hợp lệ 26.079 ảnh** (59 mẫu đã loại), **val 5.510**, **test 5.649**. Một mẫu validation có `bad_bbox` được giữ trong mẫu số đánh giá bố cục và tính là dự đoán sai. Mọi ảnh test, kể cả nhãn chuỗi không phù hợp regex, được giữ nguyên.

In [ ]:
import os,sys,json,math,time,random,hashlib,zipfile,shutil
from collections import Counter
import numpy as np
import pandas as pd
import cv2
from PIL import Image
from tqdm.auto import tqdm
import torch
from torch import nn
from torch.utils.data import Dataset,DataLoader
from torchvision import transforms
from torchvision.models import mobilenet_v3_small,MobileNet_V3_Small_Weights
from sklearn.metrics import precision_recall_fscore_support,confusion_matrix
import matplotlib.pyplot as plt

cv2.setNumThreads(0)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if REQUIRE_GPU and DEVICE.type!='cuda':
    raise RuntimeError('Cần GPU Colab để huấn luyện/đánh giá. Hãy chọn Runtime → T4 GPU.')
print('GPU:',torch.cuda.get_device_name(0) if DEVICE.type=='cuda' else 'CPU')
for path in [ZIP_PATH,MANIFEST_PATH,CRNN_CHECKPOINT,CRNN_TEST_CSV]:
    if not path.is_file(): raise FileNotFoundError(f'Thiếu dữ liệu CRNN dùng chung: {path}')

def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)

def save_json(obj,path):
    Path(path).write_text(json.dumps(obj,ensure_ascii=False,indent=2,default=str),encoding='utf8')

def atomic_torch_save(obj,dst):
    dst=Path(dst);tmp=dst.with_suffix(dst.suffix+'.tmp')
    torch.save(obj,tmp);os.replace(tmp,dst)

seed_all(SEED)
# dtype=str và sắp thứ tự như cell xuất manifest của notebook CRNN thật.
manifest=pd.read_csv(MANIFEST_PATH,dtype=str,keep_default_na=False).sort_values('image_id').reset_index(drop=True)
required={'image_id','label','label_orig','split','layout','mode','drop','input_ok','sha256','xmin','ymin','xmax','ymax'}
assert required.issubset(manifest.columns),f'Manifest thiếu {required-set(manifest.columns)}'
assert manifest.image_id.is_unique and len(manifest)==REFERENCE_EXPECTED['total']
assert set(manifest.split)=={'train','val','test'} and set(manifest.layout)==set(LABELS)
assert manifest.groupby('label_orig').split.nunique().max()==1,'Trùng nhãn nguồn xuyên split'
assert manifest[manifest.sha256!=''].groupby('sha256').split.nunique().max()==1,'Trùng SHA-256 xuyên split'
assert manifest.loc[manifest.split=='train','drop'].eq('OK').sum()==REFERENCE_EXPECTED['train']
assert manifest.split.value_counts()['val']==REFERENCE_EXPECTED['val']
assert manifest.split.value_counts()['test']==REFERENCE_EXPECTED['test']
MANIFEST_SHA=hashlib.sha256(manifest.to_csv(index=False).encode('utf8')).hexdigest()
print('Manifest SHA256:',MANIFEST_SHA)
print('Phân bố trước lọc:');display(pd.crosstab(manifest.split,manifest.layout))
print('Những mẫu val/test có crop lỗi:');display(manifest[(manifest.split!='train')&(manifest.input_ok!='1')][['image_id','split','mode']])

# CRNN checkpoint có fingerprint ghép từ chính MANIFEST_SHA; xác minh ngay từ đầu.
CRNN_CHARS=''.join(list('0123456789')+list('ABCDEFGHKLMNPSTUVXYZ')+['R'])
crnn_config_fingerprint={
 'manifest':MANIFEST_SHA,'char_order':CRNN_CHARS,'batch_size':64,
 'seed':42,'lr':1e-3,'weight_decay':1e-4,
 'w_h':(256,32),'lpr_crop':(94,24,.03,.06,'sloped',.03),
 'weighted_sampler':True,'aug':True,'model':'CRNN_cnn_bilstm2_256',
}
CRNN_IDENTITY=hashlib.sha256(json.dumps(crnn_config_fingerprint,sort_keys=True).encode()).hexdigest()
ckpt_crnn=torch.load(CRNN_CHECKPOINT,map_location='cpu',weights_only=False)
if ckpt_crnn.get('identity')!=CRNN_IDENTITY:
    raise RuntimeError('Checkpoint CRNN KHÔNG khớp manifest/tiền xử lý 85 epoch; dừng đối chứng.')
if int(ckpt_crnn['epoch'])!=REFERENCE_EXPECTED['epoch']:
    raise RuntimeError(f"CRNN không ở epoch 85: {ckpt_crnn['epoch']}")
print('CRNN checkpoint đã xác thực:',ckpt_crnn['epoch'],'epoch; identity khớp dữ liệu.')

## 3. Chuẩn bị ảnh VNLP cho MobileNet và CRNN

Sử dụng trực tiếp ảnh VNLP nguồn từ ZIP. Mẫu ảnh cắt theo bbox: **3% theo chiều ngang, 6% theo chiều dọc**, giống thí nghiệm CRNN. Trường hợp `mode=crop` giữ nguyên ảnh. Với MobileNet: crop gốc RGB → letterbox 128×128 → chuẩn hóa ImageNet. **Không tách dòng để huấn luyện bộ phân loại bố cục.**

In [ ]:
LOCAL_DATA.mkdir(parents=True,exist_ok=True)
# Tránh dùng lại cache của một ZIP khác trong cùng phiên Colab.
marker=LOCAL_DATA/'.zip_identity.json'
zip_identity={'size':ZIP_PATH.stat().st_size,'mtime_ns':ZIP_PATH.stat().st_mtime_ns,'path':str(ZIP_PATH)}
if marker.is_file():
    try: previous=json.loads(marker.read_text(encoding='utf8'))
    except Exception:previous={}
    if previous!=zip_identity:
        shutil.rmtree(LOCAL_DATA)
        LOCAL_DATA.mkdir(parents=True,exist_ok=True)
if COPY_ZIP_TO_LOCAL:
    local_zip=Path('/content/vnlp_layout_source.zip')
    if (not local_zip.is_file() or local_zip.stat().st_size!=ZIP_PATH.stat().st_size
            or not Path('/content/vnlp_layout_source_identity.json').is_file()
            or json.loads(Path('/content/vnlp_layout_source_identity.json').read_text(encoding='utf8'))!=zip_identity):
        print('Sao chép ZIP về ổ cục bộ (không thay đổi ZIP trên Drive)...')
        with ZIP_PATH.open('rb') as src,local_zip.open('wb') as dst: shutil.copyfileobj(src,dst,16*1024*1024)
        Path('/content/vnlp_layout_source_identity.json').write_text(json.dumps(zip_identity),encoding='utf8')
    source_zip=local_zip
else:source_zip=ZIP_PATH

# Kiểm tra đủ file, giải nén thiếu, không tự bỏ ảnh.
paths={}
missing=[]
for image_id in manifest.image_id:
    assert image_id.startswith('detection/') and '..' not in image_id.split('/'),f'image_id không an toàn: {image_id}'
    path=LOCAL_DATA/image_id
    paths[image_id]=str(path)
    if not path.is_file() or path.stat().st_size==0: missing.append(image_id)
print('Ảnh cần giải nén:',len(missing),'/',len(manifest))
if missing:
    with zipfile.ZipFile(source_zip) as z:
        lookup={}
        for info in z.infolist():
            name=info.filename.replace('\\','/')
            index=name.find('detection/')
            if not info.is_dir() and index>=0:lookup.setdefault(name[index:],[]).append(info)
        for image_id in tqdm(missing,desc='Giải nén dữ liệu VNLP'):
            candidates=lookup.get(image_id,[])
            if len(candidates)!=1:raise RuntimeError(f'ZIP thiếu/trùng {image_id}, count={len(candidates)}')
            dst=Path(paths[image_id]);dst.parent.mkdir(parents=True,exist_ok=True)
            temp=dst.with_name(dst.name+'.partial')
            with z.open(candidates[0]) as src,temp.open('wb') as out:shutil.copyfileobj(src,out)
            assert temp.stat().st_size==candidates[0].file_size,f'Giải nén hỏng: {image_id}'
            temp.replace(dst)
marker.write_text(json.dumps(zip_identity),encoding='utf8')
manifest['img_path']=manifest.image_id.map(paths)
manifest['bbox']=list(zip(*[manifest[c].astype(int).to_numpy() for c in ['xmin','ymin','xmax','ymax']]))
manifest['valid']=manifest.input_ok.eq('1')
train_df=manifest[(manifest.split=='train')&(manifest['drop']=='OK')].copy()
val_df=manifest[manifest.split=='val'].copy()
test_df=manifest[manifest.split=='test'].copy()
assert (len(train_df),len(val_df),len(test_df))==(26079,5510,5649)
assert train_df.valid.all(), 'Train còn ảnh không hợp lệ!'
print('Đã sẵn sàng:',len(train_df),len(val_df),len(test_df))

## 4. MobileNetV3-Small — huấn luyện riêng trên train OCR 70/15/15

Giữ quy trình huấn luyện của notebook MobileNet nguồn: ImageNet-1K V1, ảnh 128×128, batch 64, AdamW, freeze 2 epoch, macro-F1 validation, ReduceLROnPlateau, tối đa 25 epoch, patience 6. **Checkpoint lưu tên riêng và chỉ chọn theo validation.**

In [ ]:
# Tiền xử lý cắt/ghép hàng sao chép nguyên hàm từ notebook CRNN epoch 85.
# Các biến này được mã CRNN dùng khi kiểm thử; lớp MobileNet dùng cùng bbox/crop.
IMREAD_FLAGS=cv2.IMREAD_COLOR|cv2.IMREAD_IGNORE_ORIENTATION

def read_image(s):
    return cv2.imread(str(s['img_path']),IMREAD_FLAGS)

def resize_to(img, size):
    w, h = size
    interp = cv2.INTER_AREA if (img.shape[1] > w or img.shape[0] > h) else cv2.INTER_LINEAR
    return cv2.resize(img, (w, h), interpolation=interp)

def crop_box(s, img_w, img_h, is_train=False):
    xmin, ymin, xmax, ymax = s["bbox"]
    bw, bh = xmax - xmin, ymax - ymin
    if is_train:   # lệch bbox ngẫu nhiên: hơi hụt hoặc hơi rộng như bbox của YOLO
        l, r = np.random.uniform(-0.02, 0.08, 2) * bw
        t, b = np.random.uniform(-0.04, 0.14, 2) * bh
    else:
        l = r = PAD_X * bw
        t = b = PAD_Y * bh
    x0, y0 = max(0, int(round(xmin - l))), max(0, int(round(ymin - t)))
    x1, y1 = min(img_w, int(round(xmax + r))), min(img_h, int(round(ymax + b)))
    return x0, y0, x1, y1

def get_plate_crop(img, s, is_train=False):
    if s["mode"] == "crop":
        return img
    h, w = img.shape[:2]
    x0, y0, x1, y1 = crop_box(s, w, h, is_train)
    crop = img[y0:y1, x0:x1]
    return crop if crop.size > 0 else img

def find_row_gap(crop):
    """Tìm đường thẳng (có thể nghiêng) giữa 2 dòng chữ, cắt qua ít nét mực nhất.
    Trả về (y ở mép trái, y ở mép phải) theo pixel của crop và cờ tin cậy."""
    h, w = crop.shape[:2]
    if h < 12 or w < 12:
        return (h / 2, h / 2), False
    sh = 96
    sw = min(256, max(64, round(w * sh / h)))
    gray = cv2.cvtColor(cv2.resize(crop, (sw, sh), interpolation=cv2.INTER_AREA), cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (3, 3), 0)
    ink = (cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 31, 7) > 0).astype(np.float32)
    xs = np.arange(round(0.12 * sw), round(0.88 * sw))               # bỏ viền / ốc vít hai bên khi chấm điểm
    cc, dd = np.meshgrid(np.arange(round(0.30 * sh), round(0.70 * sh) + 1),
                         np.linspace(-0.35 * sh, 0.35 * sh, 15), indexing="ij")
    cc, dd = cc.ravel().astype(np.float32), dd.ravel().astype(np.float32)
    ys = cc[:, None] + dd[:, None] * (xs[None, :] / max(1, sw - 1) - 0.5)

    def line_ink(y):
        return np.mean([ink[np.clip(np.rint(y + band).astype(int), 0, sh - 1), xs[None, :]].mean(axis=1)
                        for band in (-1, 0, 1)], axis=0)

    gap = line_ink(ys)
    score = gap + 0.04 * np.abs(cc / sh - 0.5) + 0.003 * np.abs(dd) / (0.35 * sh)   # hơi ưu tiên giữa và ít nghiêng
    b = int(np.argmin(score))
    above = max(float(line_ink(ys[b:b + 1] - o * sh)[0]) for o in (0.12, 0.18, 0.24))
    below = max(float(line_ink(ys[b:b + 1] + o * sh)[0]) for o in (0.12, 0.18, 0.24))
    ok = min(above, below) > 0.03 and gap[b] / max(1e-6, min(above, below)) < 0.65   # 2 phía đều có chữ, khe rõ
    y_left, y_right = (cc[b] - dd[b] / 2) * h / sh, (cc[b] + dd[b] / 2) * h / sh
    return (float(np.clip(y_left, 2, h - 2)), float(np.clip(y_right, 2, h - 2))), bool(ok)

def row_strip(crop, y_top, y_bot):
    """Nắn dải giữa 2 đường y_top, y_bot (mỗi cột một giá trị) thành dải thẳng, rồi resize về 94x24."""
    w = crop.shape[1]
    out_h = max(2, int(round(float(np.mean(y_bot - y_top)))))
    t = np.linspace(0, 1, out_h, dtype=np.float32)[:, None]
    map_y = (y_top[None, :] + t * (y_bot - y_top)[None, :]).astype(np.float32)
    map_x = np.tile(np.arange(w, dtype=np.float32), (out_h, 1))
    strip = cv2.remap(crop, map_x, map_y, cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)
    return resize_to(strip, (IMG_W, IMG_H))

def row_cut(crop):
    h, w = crop.shape[:2]
    if SPLIT_METHOD == "sloped":
        (y_left, y_right), ok = find_row_gap(crop)
    else:
        (y_left, y_right), ok = (h / 2, h / 2), True
    return np.linspace(y_left, y_right, w, dtype=np.float32), ok

def split_two_rows(crop, overlap=SPLIT_OVERLAP):
    # mỗi dòng được nắn thẳng và resize riêng về 94x24 như một biển 1 dòng, rồi ghép ngang -> 188x24
    h, w = crop.shape[:2]
    cut, _ = row_cut(crop)
    ov = overlap * h
    top = row_strip(crop, np.zeros(w, np.float32), np.minimum(cut + ov, h - 1))
    bot = row_strip(crop, np.maximum(cut - ov, 0), np.full(w, h - 1, np.float32))
    return np.concatenate((top, bot), axis=1)

In [ ]:
def crnn_letterbox(image):
    if image is None or image.size==0:raise ValueError('Ảnh đầu vào rỗng')
    h,w=image.shape[:2]
    scale=min(CRNN_H/h,CRNN_W/w)
    neww,newh=max(1,int(round(w*scale))),max(1,int(round(h*scale)))
    small=cv2.resize(image,(neww,newh),interpolation=cv2.INTER_AREA if scale<1 else cv2.INTER_LINEAR)
    out=np.full((CRNN_H,CRNN_W,3),127,np.uint8)
    y=(CRNN_H-newh)//2
    out[y:y+newh,:neww]=small
    return out

In [ ]:
CLASS_ID={'one_row':0,'two_rows':1}
IMAGENET_NORMALIZE=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((.485,.456,.406),(.229,.224,.225)),
])
AUGMENT=transforms.Compose([
    transforms.RandomRotation(5,expand=True,fill=(127,127,127)),
    transforms.ColorJitter(brightness=.2,contrast=.2,saturation=.1),
    transforms.RandomApply([transforms.GaussianBlur(3,sigma=(.1,.6))],p=.1),
])

def classifier_letterbox(image):
    image=image.convert('RGB');w,h=image.size
    scale=min(128/w,128/h)
    nw,nh=max(1,round(w*scale)),max(1,round(h*scale))
    image=image.resize((nw,nh),Image.Resampling.BILINEAR)
    canvas=Image.new('RGB',(128,128),(127,127,127))
    canvas.paste(image,((128-nw)//2,(128-nh)//2))
    return canvas

def make_layout_tensor(crop_bgr):
    rgb=cv2.cvtColor(crop_bgr,cv2.COLOR_BGR2RGB)
    return IMAGENET_NORMALIZE(classifier_letterbox(Image.fromarray(rgb)))

class LayoutDataset(Dataset):
    def __init__(self,df,training=False):
        self.samples=df.to_dict('records');self.training=training
    def __len__(self):return len(self.samples)
    def __getitem__(self,i):
        s=self.samples[i]
        if not s['valid']:
            image=Image.new('RGB',(128,128),(127,127,127)); valid=False
        else:
            img=read_image(s)
            if img is None:raise RuntimeError(f'Không đọc được ảnh sau audit: {s["image_id"]}')
            crop=get_plate_crop(img,s,is_train=False)
            if crop.size==0:raise RuntimeError(f'Crop rỗng: {s["image_id"]}')
            image=Image.fromarray(cv2.cvtColor(crop,cv2.COLOR_BGR2RGB)); valid=True
            if self.training: image=AUGMENT(image)
        return IMAGENET_NORMALIZE(classifier_letterbox(image)),CLASS_ID[s['layout']],valid,s['image_id']

def loader_for(df,training=False,epoch=0):
    return DataLoader(LayoutDataset(df,training),batch_size=BATCH_SIZE,
      shuffle=training,num_workers=NUM_WORKERS,pin_memory=DEVICE.type=='cuda',
      generator=torch.Generator().manual_seed(SEED+epoch*1009),
      persistent_workers=NUM_WORKERS>0)

@torch.inference_mode()
def evaluate_layout(model,df):
    model.eval();rows=[]
    for images,y,valid,ids in tqdm(loader_for(df),desc='Đánh giá MobileNet',leave=False):
        logits=model(images.to(DEVICE))
        probs=logits.softmax(1)
        conf,pred=probs.max(1)
        pred=pred.cpu().tolist();conf=conf.cpu().tolist()
        for iid,yy,vv,pp,cc in zip(ids,y.tolist(),valid.tolist(),pred,conf):
            rows.append({'image_id':iid,'target':yy,'prediction':pp if vv else -1,
                         'confidence':cc if vv else None,'valid':bool(vv)})
    details=pd.DataFrame(rows)
    assert len(details)==len(df) and details.image_id.is_unique
    metrics=layout_metrics(details)
    return metrics,details

def layout_metrics(d):
    # accuracy: invalid crop counted wrong; macro F1 with classes 0 and 1
    p,r,f,s=precision_recall_fscore_support(d.target,d.prediction,labels=[0,1],zero_division=0)
    return {'accuracy':float((d.target==d.prediction).mean()),
            'macro_f1':float(f.mean()),'samples':int(len(d)),
            'invalid_inputs':int((~d.valid).sum()),
            'f1_one_row':float(f[0]),'f1_two_rows':float(f[1])}

BEST=OUTPUT_DIR/'layout_best.pt'
LAST=OUTPUT_DIR/'layout_last.pt'
HISTORY=OUTPUT_DIR/'history_mobilenet.csv'
CLASSIFIER_IDENTITY=hashlib.sha256(json.dumps({
  'manifest_sha':MANIFEST_SHA,'train_rule':'split=train and drop=OK',
  'val_rule':'split=val including invalid','class_order':LABELS,
  'crop':(PAD_X,PAD_Y,'mode_crop_keeps_full'),
  'image_size':128,'init':'IMAGENET1K_V1','seed':SEED,
  'batch':BATCH_SIZE,'max_epochs':MAX_EPOCHS,'warmup':WARMUP_EPOCHS,
  'optimizer':'AdamW weight_decay=1e-4','scheduler':'plateau_max_macrof1',
},sort_keys=True).encode()).hexdigest()

def init_classifier(pretrained):
    net=mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1 if pretrained else None)
    net.classifier[-1]=nn.Linear(net.classifier[-1].in_features,2)
    return net.to(DEVICE).float()

In [ ]:
def run_layout_train_epoch(model,optimizer,epoch):
    frozen=epoch<=WARMUP_EPOCHS
    model.train()
    model.features.requires_grad_(not frozen)
    if frozen:model.features.eval()  # giữ batch-statistics trong giai đoạn freeze
    total_loss=0.;correct=0;count=0
    criterion=nn.CrossEntropyLoss()
    for x,y,valid,ids in tqdm(loader_for(train_df,training=True,epoch=epoch),desc=f'Train layout {epoch}',leave=False):
        if not bool(valid.all()):raise RuntimeError('Mẫu train không hợp lệ')
        x=x.to(DEVICE);y=y.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits=model(x);loss=criterion(logits,y)
        if not bool(torch.isfinite(loss)):raise RuntimeError('Training loss NaN/Inf')
        loss.backward();nn.utils.clip_grad_norm_(model.parameters(),max_norm=5.,error_if_nonfinite=True)
        optimizer.step()
        n=len(y);count+=n;total_loss+=float(loss.detach())*n
        correct+=int((logits.detach().argmax(1)==y).sum())
    return {'train_loss':total_loss/count,'train_acc':correct/count}

seed_all(SEED)
resume=LAST.is_file()
model=init_classifier(pretrained=TRAIN_IF_NEEDED and not resume and not BEST.is_file())
optimizer=torch.optim.AdamW([
  {'params':model.features.parameters(),'lr':1e-5},
  {'params':model.classifier.parameters(),'lr':1e-3},
],weight_decay=1e-4)
scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode='max',factor=.5,patience=2,min_lr=1e-7)
start_epoch=1;history=[];best_key=(-1.,-1.);stale=0
if resume:
    st=torch.load(LAST,map_location='cpu',weights_only=False)
    assert st['identity']==CLASSIFIER_IDENTITY,'Không thể resume checkpoint khác dữ liệu'
    model.load_state_dict(st['model']);optimizer.load_state_dict(st['optimizer'])
    scheduler.load_state_dict(st['scheduler']);start_epoch=st['epoch']+1
    history=st['history'];best_key=tuple(st['best_key']);stale=st['stale']
    print('Resume MobileNet từ epoch',start_epoch)
elif BEST.is_file():
    st=torch.load(BEST,map_location='cpu',weights_only=False)
    assert st['identity']==CLASSIFIER_IDENTITY,'Best checkpoint thuộc phép chia khác'
    model.load_state_dict(st['model'])
    print('Đã có best checkpoint, dùng kết quả cũ; không ghi đè')
    TRAIN_IF_NEEDED=False
if TRAIN_IF_NEEDED and stale<EARLY_STOP_PATIENCE:
    for epoch in range(start_epoch,MAX_EPOCHS+1):
        seed_all(SEED+epoch)
        if epoch==WARMUP_EPOCHS+1:
            optimizer.param_groups[0]['lr']=1e-5
            optimizer.param_groups[1]['lr']=1e-4
        train_stats=run_layout_train_epoch(model,optimizer,epoch)
        val_metrics,val_preds=evaluate_layout(model,val_df)
        key=(val_metrics['macro_f1'],val_metrics['accuracy'])
        improved=key>best_key
        if improved:best_key=key;stale=0
        else:stale+=1
        scheduler.step(val_metrics['macro_f1'])
        history.append({'epoch':epoch,**train_stats,
           'val_acc':val_metrics['accuracy'],'val_macro_f1':val_metrics['macro_f1'],
           'backbone_lr':optimizer.param_groups[0]['lr'],
           'classifier_lr':optimizer.param_groups[1]['lr']})
        state={'identity':CLASSIFIER_IDENTITY,'epoch':epoch,'model':model.state_dict(),
          'optimizer':optimizer.state_dict(),'scheduler':scheduler.state_dict(),
          'history':history,'best_key':best_key,'stale':stale,'val_metrics':val_metrics}
        if improved:
            atomic_torch_save(state,BEST)
            val_preds.to_csv(OUTPUT_DIR/'best_val_predictions.csv',index=False)
        atomic_torch_save(state,LAST)
        pd.DataFrame(history).to_csv(HISTORY,index=False)
        print(f"Epoch {epoch:02d}: train_acc={train_stats['train_acc']:.4f} | val_acc={val_metrics['accuracy']:.4f} | val_F1={val_metrics['macro_f1']:.4f} {'*' if improved else ''}")
        if stale>=EARLY_STOP_PATIENCE:
            print('Đã dừng sớm theo validation');break
if not BEST.is_file():
    raise FileNotFoundError('Chưa có MobileNet best checkpoint. Đặt TRAIN_IF_NEEDED=True rồi chạy lại.')
best_state=torch.load(BEST,map_location='cpu',weights_only=False)
assert best_state['identity']==CLASSIFIER_IDENTITY
model.load_state_dict(best_state['model']);model.eval()
print('MobileNet best epoch:',best_state['epoch'],'| val:',best_state['val_metrics'])

## 5. Cố định ngưỡng hình học bằng validation và kiểm thử MobileNet

Ngưỡng được chọn tối đa **macro-F1 của bố cục trên validation** (phụ: Accuracy), không dùng OCR Accuracy của tập test để chỉnh. Sau khi chốt cả ngưỡng và MobileNet checkpoint mới đánh giá trên **5.649 test**.

In [ ]:
def bbox_ratio(df):
    w=(df.xmax.astype(float)-df.xmin.astype(float)).to_numpy()
    h=(df.ymax.astype(float)-df.ymin.astype(float)).to_numpy()
    if (h<=0).any():raise ValueError('BBox có chiều cao không dương')
    return w/h

def choose_ratio_threshold(validation):
    # Phân loại two_rows nếu bbox width/height < threshold.
    good=validation[validation.valid]
    ratios=bbox_ratio(good)
    y=(good.layout=='two_rows').astype(int).to_numpy()
    ix=np.argsort(ratios,kind='mergesort')
    r=ratios[ix];yy=y[ix]
    boundaries=np.r_[0,np.flatnonzero(r[1:]!=r[:-1])+1,len(r)]
    positive=np.r_[0,np.cumsum(yy)]
    tp=positive[boundaries].astype(float)
    fp=boundaries.astype(float)-tp
    fn=float(yy.sum())-tp
    tn=float((1-yy).sum())-fp
    f1_pos=np.divide(2*tp,2*tp+fp+fn,out=np.zeros_like(tp),where=(2*tp+fp+fn)>0)
    f1_neg=np.divide(2*tn,2*tn+fp+fn,out=np.zeros_like(tp),where=(2*tn+fp+fn)>0)
    macro=(f1_pos+f1_neg)/2
    acc=(tp+tn)/len(y)
    # ngưỡng ứng với r < t; tránh chọn trực tiếp một điểm đúng bằng r của ảnh.
    candidate=np.empty(len(boundaries))
    candidate[0]=np.nextafter(r[0],-np.inf)
    candidate[-1]=np.nextafter(r[-1],np.inf)
    for j in range(1,len(boundaries)-1):
        at=boundaries[j]
        candidate[j]=(r[at-1]+r[at])/2
    # Tie-break: accuracy cao hơn, rồi ngưỡng nhỏ hơn.
    rank=np.lexsort((candidate,-acc,-macro))
    best=int(rank[0]); t=float(candidate[best])
    return t,{'threshold':t,'val_macro_f1':float(macro[best]),
              'val_accuracy_valid_only':float(acc[best]),'valid_validation_samples':len(y),
              'invalid_validation_samples':int((~validation.valid).sum()),
              'selected_on':'validation layout macro-F1; tie accuracy',
              'decision_rule':'two_rows if bbox_width / bbox_height < threshold'}

threshold,threshold_protocol=choose_ratio_threshold(val_df)
save_json(threshold_protocol,OUTPUT_DIR/'geometry_threshold_validation.json')
print('Ngưỡng hình học chốt bằng validation:',threshold)
print('Val macro-F1 của rule (chỉ ảnh hợp lệ):',threshold_protocol['val_macro_f1'])

layout_test_metrics,layout_test_predictions=evaluate_layout(model,test_df)
layout_test_predictions.to_csv(OUTPUT_DIR/'layout_test_predictions.csv',index=False)
save_json(layout_test_metrics,OUTPUT_DIR/'layout_test_metrics.json')
conf=confusion_matrix(layout_test_predictions.target,layout_test_predictions.prediction,labels=[0,1,-1])
display(pd.DataFrame(conf,index=['true_one_row','true_two_rows','true_invalid'],
    columns=['pred_one_row','pred_two_rows','pred_invalid']))
print('MobileNet mới test 70/15/15:',layout_test_metrics)

## 6. CRNN 85 epoch — nạp chính xác kiến trúc và tiền xử lý

Dùng kiến trúc CNN → 2-layer BiLSTM → CTC 32 lớp của notebook CRNN. **Không dùng mô hình/đầu vào CRNN cũ 37 lớp**. Nhánh `reference` phải tái tạo dự đoán đã lưu trước khi cho phép kết luận ba nhánh.

In [ ]:
# Bản sao chính xác class CRNN trong notebook epoch 85.
CHAR2ID={ch:i+1 for i,ch in enumerate(CRNN_CHARS)}
ID2CHAR={i+1:ch for i,ch in enumerate(CRNN_CHARS)}
BLANK=0
NUM_CLASSES=len(CRNN_CHARS)+1

class CRNN(nn.Module):
    """Kiến trúc CRNN gốc của nhóm: CNN -> 2-layer BiLSTM -> Linear -> CTC."""
    def __init__(self,nclass=NUM_CLASSES,hidden=256):
        super().__init__()
        def block(a,b):return [nn.Conv2d(a,b,3,padding=1,bias=False),nn.BatchNorm2d(b),nn.ReLU(inplace=True)]
        self.cnn=nn.Sequential(
            *block(3,64),nn.MaxPool2d(2,2),
            *block(64,128),nn.MaxPool2d(2,2),
            *block(128,256),*block(256,256),nn.MaxPool2d((2,1),(2,1)),
            *block(256,512),*block(512,512),nn.MaxPool2d((2,1),(2,1)),
            nn.Conv2d(512,512,(2,1),bias=False),nn.BatchNorm2d(512),nn.ReLU(inplace=True))
        self.rnn=nn.LSTM(512,hidden,num_layers=2,batch_first=True,bidirectional=True,dropout=.1)
        self.head=nn.Linear(hidden*2,nclass)
        self.apply(self._init)
    @staticmethod
    def _init(m):
        if isinstance(m,nn.Conv2d):nn.init.kaiming_normal_(m.weight,nonlinearity='relu')
        elif isinstance(m,nn.Linear):nn.init.xavier_uniform_(m.weight);nn.init.zeros_(m.bias)
    def forward(self,x):
        f=self.cnn(x)
        assert f.shape[2]==1, f'Feature height != 1: {f.shape}'
        seq=f.squeeze(2).transpose(1,2)
        y,_=self.rnn(seq)
        return self.head(y).transpose(0,1)


crnn=CRNN(NUM_CLASSES).to(DEVICE).float()
crnn.load_state_dict(ckpt_crnn['model'],strict=True)
crnn.eval()
assert sum(p.numel() for p in crnn.parameters())==8198240
print('CRNN đã nạp: epoch',ckpt_crnn['epoch'],'| params',sum(p.numel() for p in crnn.parameters()))
del ckpt_crnn

## 7. So sánh ba nhánh trên **cùng 5.649 ảnh test**

Ảnh được đọc **BGR** bằng `IMREAD_IGNORE_ORIENTATION`; bbox, phép tìm khe nghiêng, tách hai dải và ghép ngang được dùng **đúng theo code CRNN 85 epoch**. Kết quả được so sánh một-một với file `crnn_test_predictions.csv`, không chấp nhận sai khác tự phát.

In [ ]:
@torch.inference_mode()
def decode_crnn(logits):
    ids=logits.detach().argmax(2).transpose(0,1).cpu().tolist()
    preds=[]
    for seq in ids:
        out=[];last=None
        for idx in seq:
            if idx!=0 and idx!=last:out.append(ID2CHAR[idx])
            last=idx
        preds.append(''.join(out))
    return preds

def ocr_tensor_from_raw(img,s,layout):
    if not s['valid'] or img is None:
        return None
    crop=get_plate_crop(img,s,is_train=False)
    if layout=='two_rows':
        step=split_two_rows(crop)  # EXACT CRNN85 with overlap 0.03, no row-confidence fallback
    elif layout=='one_row':
        step=resize_to(crop,(IMG_W,IMG_H))
    else:raise ValueError(layout)
    x=crnn_letterbox(step)
    return torch.from_numpy(np.ascontiguousarray(x.transpose(2,0,1))).float().div_(127.5).sub_(1.)

def edit_distance(a,b):
    prev=list(range(len(b)+1))
    for i,ca in enumerate(a,1):
        cur=[i]
        for j,cb in enumerate(b,1):cur.append(min(cur[-1]+1,prev[j]+1,prev[j-1]+(ca!=cb)))
        prev=cur
    return prev[-1]

# Không dùng ground truth test để chọn hyperparameter ở các bước trước.
layout_lookup=layout_test_predictions.set_index('image_id')
assert set(layout_lookup.index)==set(test_df.image_id)

@torch.inference_mode()
def evaluate_three_ocr_routes():
    records=[]; n=len(test_df)
    crnn.eval();model.eval()
    rows=test_df.to_dict('records')
    for offset in tqdm(range(0,n,EVAL_BATCH),desc='Ba phương án layout → CRNN'):
        part=rows[offset:offset+EVAL_BATCH]
        local=[];jobs=[];raw_images={};classification_inputs=[];classification_indices=[]
        # Đọc/cắt ảnh một lần, MobileNet chạy theo batch để giảm chi phí GPU.
        for ix,s in enumerate(part):
            item={'image_id':s['image_id'],'gt':s['label'],'reference_layout':s['layout'],
                  'bbox_ratio':float((int(s['xmax'])-int(s['xmin']))/(int(s['ymax'])-int(s['ymin']))),
                  'crop_valid':bool(s['valid'])}
            item['geometry_layout']='two_rows' if item['bbox_ratio']<threshold else 'one_row'
            raw=read_image(s) if s['valid'] else None
            if s['valid'] and raw is None:raise RuntimeError('Không đọc được ảnh '+s['image_id'])
            raw_images[ix]=raw
            if s['valid']:
                crop=get_plate_crop(raw,s,is_train=False)
                if crop.size==0:raise RuntimeError('Crop rỗng: '+s['image_id'])
                classification_indices.append(ix)
                classification_inputs.append(make_layout_tensor(crop))
            else:
                item['mobilenet_layout']='invalid';item['mobilenet_confidence']=None
            local.append(item)
        if classification_indices:
            scores=model(torch.stack(classification_inputs).to(DEVICE)).softmax(1)
            confidences,classes=scores.max(1)
            for ix,cls,conf in zip(classification_indices,classes.cpu().tolist(),confidences.cpu().tolist()):
                local[ix]['mobilenet_layout']=LABELS[int(cls)]
                local[ix]['mobilenet_confidence']=float(conf)
        # Cùng một ảnh chỉ xử lý tối đa hai lựa chọn bố cục khác nhau.
        for ix,s in enumerate(part):
            item=local[ix];raw=raw_images[ix]
            for layout in set([item['reference_layout'],item['geometry_layout'],item['mobilenet_layout']]):
                if layout not in LABELS:continue
                tensor=ocr_tensor_from_raw(raw,s,layout)
                if tensor is not None:jobs.append((ix,layout,tensor))
        # Chuỗi suy luận batched; không chọn checkpoint/threshold ở cell này.
        predictions={}
        for j in range(0,len(jobs),EVAL_BATCH):
            subset=jobs[j:j+EVAL_BATCH]
            out=crnn(torch.stack([r[2] for r in subset]).to(DEVICE))
            for (ix,layout,_),pred in zip(subset,decode_crnn(out)):
                predictions[ix,layout]=pred
        for ix,item in enumerate(local):
            for key,kind in [('reference','reference_layout'),('geometry','geometry_layout'),('mobilenet','mobilenet_layout')]:
                layout=item[kind]
                item[key+'_prediction']=predictions.get((ix,layout),'')
                item[key+'_correct']=item[key+'_prediction']==item['gt']
                item[key+'_layout_correct']=(layout==item['reference_layout'])
            records.append(item)
    details=pd.DataFrame(records)
    assert len(details)==5649 and details.image_id.is_unique
    assert set(details.image_id)==set(test_df.image_id)
    return details

routes=evaluate_three_ocr_routes()
# Bắt buộc kiểm tra với kết quả CRNN gốc, trên TOÀN BỘ 5649 ảnh.
reference=pd.read_csv(CRNN_TEST_CSV,dtype=str,keep_default_na=False)
assert reference.image_id.is_unique and len(reference)==5649
comparison=routes[['image_id','gt','reference_prediction']].merge(
    reference[['image_id','gt','pred_greedy']],on='image_id',validate='one_to_one',suffixes=('','_saved'))
assert len(comparison)==5649 and comparison.image_id.is_unique
wrong_gt=comparison[comparison.gt!=comparison.gt_saved]
wrong_pred=comparison[comparison.reference_prediction!=comparison.pred_greedy]
if len(wrong_gt) or len(wrong_pred):
    mismatch=comparison[(comparison.gt!=comparison.gt_saved)|(comparison.reference_prediction!=comparison.pred_greedy)]
    mismatch.to_csv(OUTPUT_DIR/'REFERENCE_MISMATCH_DEBUG.csv',index=False)
    raise RuntimeError(f'CRNN reference KHÔNG trùng kết quả đã lưu: {len(wrong_gt)} nhãn khác, {len(wrong_pred)} prediction khác. Xem REFERENCE_MISMATCH_DEBUG.csv và kiểm tra tiền xử lý/ZIP/phiên bản thư viện, KHÔNG báo cáo Bảng 4.7.')
assert int(routes.reference_correct.sum())==5528,'Reference đúng không phải 5.528 ảnh'
print('ĐÃ XÁC MINH: 5.649 ID, tất cả GT, tất cả dự đoán CRNN reference khớp checkpoint gốc.')

## 8. Bảng 4.7 — Sequence Accuracy, CER và phân tích sai số

Các chỉ số **được tính từ dự đoán thực tế**, không suy ra từ accuracy của MobileNet. CER = tổng Levenshtein / tổng ký tự nhãn. Lưu đầy đủ CSV để có thể truy vết từng mẫu.

In [ ]:
def measure_route(df,branch):
    preds=df[branch+'_prediction']
    edits=sum(edit_distance(a,b) for a,b in zip(df.gt,preds))
    denom=sum(len(a) for a in df.gt)
    return {'method':branch,'samples':len(df),'correct':int((df.gt==preds).sum()),
            'sequence_accuracy':float((df.gt==preds).mean()),
            'cer':float(edits/denom),
            'layout_correct':int(df[branch+'_layout_correct'].sum()),
            'layout_accuracy':float(df[branch+'_layout_correct'].mean()),
            'empty_predictions':int((preds=='').sum())}

summary=pd.DataFrame([measure_route(routes,name) for name in ['reference','geometry','mobilenet']])
assert len(summary)==3
ref_cer=float(summary.iloc[0].cer)
print('BẢNG 4.7 — Các tỷ lệ theo %:')
table_47=summary.assign(
    **{'Sequence Accuracy (%)':(summary.sequence_accuracy*100).round(4),
       'CER (%)':(summary.cer*100).round(4),
       'Accuracy bố cục (%)':(summary.layout_accuracy*100).round(4)}
)[['method','samples','correct','Sequence Accuracy (%)','CER (%)','Accuracy bố cục (%)']]
display(table_47)

transition_rows=[]
for branch in ['geometry','mobilenet']:
    a=routes.reference_correct;b=routes[branch+'_correct']
    changed=routes[branch+'_layout_correct']==False
    transition_rows.append({'method':branch,
      'both_correct':int((a&b).sum()),'reference_only_correct':int((a&~b).sum()),
      'branch_only_correct':int((~a&b).sum()),'both_wrong':int((~a&~b).sum()),
      'layout_wrong':int(changed.sum()),
      'layout_wrong_reference_correct_branch_wrong':int((changed&a&~b).sum()),
      'layout_wrong_reference_wrong_branch_correct':int((changed&~a&b).sum())})
transitions=pd.DataFrame(transition_rows)
print('Thay đổi đúng/sai so với nhãn bố cục tham chiếu:');display(transitions)
for branch in ['geometry','mobilenet']:
    conf=pd.crosstab(routes.reference_layout,routes[branch+'_layout']).reindex(
        index=LABELS,columns=LABELS+['invalid'],fill_value=0)
    print('Ma trận nhầm lẫn bố cục',branch);display(conf)
    conf.to_csv(OUTPUT_DIR/f'layout_confusion_{branch}.csv')

routes.to_csv(OUTPUT_DIR/'chapter4_5_three_route_predictions.csv',index=False)
summary.to_csv(OUTPUT_DIR/'chapter4_5_summary.csv',index=False)
table_47.to_csv(OUTPUT_DIR/'chapter4_5_table_4_7.csv',index=False,encoding='utf-8-sig')
transitions.to_csv(OUTPUT_DIR/'chapter4_5_transitions.csv',index=False)
protocol={'experiment':'VNLP reference/geometry/MobileNet -> CRNN85 greedy',
    'manifest_sha256':MANIFEST_SHA,'crnn_identity':CRNN_IDENTITY,
    'crnn_epoch':85,'mobile_checkpoint':str(BEST),
    'mobile_best_epoch':int(best_state['epoch']),
    'sample_counts':{'train':26079,'val':5510,'test':5649},
    'geometry_threshold_selected_on_val':threshold_protocol,
    'reference_exact_reproduction':True,
    'metrics':summary.to_dict('records'),
    'split_integrity':'manifest original label_orig and sha256 nonoverlap across split',
    'different_modalities':'MobileNet input RGB ImageNet normalized; CRNN BGR [-1,1]',
    'limitations':'BBox from ground truth; no YOLO detection/video or speed claim'}
save_json(protocol,OUTPUT_DIR/'chapter4_5_protocol.json')
print('ĐÃ XUẤT:',OUTPUT_DIR)

## 9. Hình minh họa tùy chọn cho Chương 4.5

Chỉ lấy **ảnh thật từ VNLP test** mà một hoặc cả hai phương pháp bố cục dự đoán sai. Lưu hình minh họa được chụp từ chính ảnh và bản ghi OCR có đối chứng; không dùng ảnh minh họa ngoài tập test.

In [ ]:
from matplotlib import pyplot as plt
errors=routes[(~routes.geometry_layout_correct)|(~routes.mobilenet_layout_correct)]
errors=errors.sort_values('image_id').head(6)
if len(errors)==0:
    print('Không có trường hợp phân loại bố cục sai trên test; không tạo ảnh minh họa lỗi.')
else:
    rows_by_id=test_df.set_index('image_id')
    fig,ax=plt.subplots(len(errors),1,figsize=(10,max(3,2.5*len(errors))),squeeze=False)
    for i,e in enumerate(errors.itertuples(index=False)):
        s=rows_by_id.loc[e.image_id].to_dict();s['img_path']=rows_by_id.loc[e.image_id,'img_path']
        crop=get_plate_crop(read_image(s),s,is_train=False)
        ax[i,0].imshow(cv2.cvtColor(crop,cv2.COLOR_BGR2RGB))
        ax[i,0].set_title(f"GT: {e.gt} | Ref: {e.reference_prediction} | Hình học: {e.geometry_prediction} | MobileNet: {e.mobilenet_prediction}\nLayout: {e.reference_layout} / {e.geometry_layout} / {e.mobilenet_layout}",fontsize=9)
        ax[i,0].axis('off')
    plt.tight_layout()
    path=OUTPUT_DIR/'chapter4_5_layout_error_examples.png'
    fig.savefig(path,dpi=160,bbox_inches='tight')
    plt.show()
    print('Ảnh mẫu thực tế:',path)

## 10. Các tệp dùng cho Mục 4.5

- **`chapter4_5_table_4_7.csv`**: số liệu 3 dòng cần chuyển vào Bảng 4.7 của báo cáo.
- **`chapter4_5_three_route_predictions.csv`**: dự đoán từng ảnh ở cả 3 phương án, dùng kiểm tra nguyên nhân lỗi.
- **`chapter4_5_transitions.csv`**: số trường hợp từ đúng thành sai / từ sai thành đúng.
- **`geometry_threshold_validation.json`**: ngưỡng W/H đã chốt trên validation.
- **`layout_best.pt`** và **`history_mobilenet.csv`**: trọng số, lịch sử huấn luyện MobileNet 70/15/15; KHÔNG phải MobileNet gốc 80/10/10.
- **`chapter4_5_protocol.json`**: xác nhận đã tái tạo đúng nhánh reference CRNN 85 epoch và thông tin giao thức.
- **`chapter4_5_layout_error_examples.png`**: minh họa từ ảnh test thật (nếu có sai bố cục).

**Phạm vi kết quả:** trên crop có bbox tham chiếu; không phải OCR end-to-end trên video. Không dùng test để chọn MobileNet checkpoint hoặc ngưỡng hình học.